# Проект 1.
# Бизнес-вопрос: Окупилось ли подорожание 1 марта 2026 года

С 1 марта 2026 года тариф «Базовый» подорожал с 199 до 249 ₽, «Расширенный» - с 399 до 449 ₽. Нужно понять, привело ли увеличение стоимости тарифа к увеличению дохода, или отток клиентов съел всю разницу. Отличался ли эффект для двух разных тарифов?

### Гипотезы

**Гипотеза 1.** 

Рост цен на тарифы увеличил отток, база активных пользователей сократилась, а прибыль снизилась.

**Гипотеза 2.**

В процентном соотношении, цена "Базового" увеличилась вдвое больше, чем цена "Расширенного" - +25% и +12,5% соответственно. Можно предположить, что отток клиентов "Базового" был больше - они отказались от подписки или перешли на "Расширенный".


---
## 1. Подключение и параметры

In [ ]:
import math

import matplotlib.pyplot as plt
import pandas as pd
from sqlalchemy import text

from prime.config import get_engine   

engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            # Сутки и месяцы считаем по UTC — так они у всех одинаковые.
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


print("подключились как:", q("select current_user as login").iloc[0, 0])

In [ ]:
# Все параметры анализа — в одном месте. Поменяли здесь — пересчиталось всё ниже.

TARIFF = {1: "Базовый", 2: "Расширенный"}          
COLORS = {"Базовый": "#C1272D", "Расширенный": "#9CA3AF"}   

PRICE_CHANGE = "2026-03-01"                       

# Окна сравнения: по три месяца до и после подорожания. Правый конец не включён.
BEFORE = ("2025-12-01", "2026-03-01")             # декабрь, январь, февраль
AFTER = ("2026-03-01", "2026-06-01")              # март, апрель, май

# Длинный ряд для графиков: с сентября 2025 по июль 2026 (месяцы, от первого до последнего).
FIRST_MONTH, LAST_MONTH = "2025-09-01", "2026-07-01"

###  Почему такие окна

Подорожание случилось 01.03.2026, поэтому для анализа беру два временных промежутка по 3 месяца каждый: с 01.12 до 01.03 (конец не включен) и с 01.03 до 01.06. Важно отметить, что периоды не абсолютно равны по продолжительности. Из предудущего ДЗ мы знаем, что в данных за апрель нет двух дней, а сами месяцы разной продолжительности. Поэтому все данные будем сравнивать в расчете на день. 


---
## 2. Аудит данных

Проверим две вещи, чтобы предположения, которые мы сделали при постановке гипотезы, были подтверждены числами.

**Есть ли дни без платежей?** - в ДЗ-2 в апреле не хватало двух дней.

**Действительно ли цена изменилась 1 марта?** - сравниваем суммы, которые пользователи реально платили по двум тарифам


In [ ]:
# Успешные платежи по дням за весь длинный ряд.
days = q("""
    select paid_at::date as day, count(*) as n_payments
    from prime.payments
    where status = 'success'
      and paid_at >= cast(:a as timestamptz)
      and paid_at <  cast(:b as timestamptz) + interval '1 month'
    group by 1
    order by 1
""", a=FIRST_MONTH, b=LAST_MONTH)

days["day"] = pd.to_datetime(days["day"])

# Все календарные дни периода и те из них, которых нет в платежах.
calendar = pd.date_range(FIRST_MONTH, pd.Timestamp(LAST_MONTH) + pd.offsets.MonthEnd(0), freq="D")
missing = calendar.difference(days["day"])
print("дней в календаре:", len(calendar), "| дней с платежами:", len(days))
print("дни без единого платежа:", [d.strftime("%Y-%m-%d") for d in missing])

# Сколько дней с данными в каждом месяце
days["month"] = days["day"].dt.strftime("%Y-%m")
days_with_data = days.groupby("month").size().rename("days_data")
days_with_data

In [ ]:
# Какие суммы платили по каждому тарифу до и после 1 марта.
# Одна строка — тариф × период × сумма платежа; n — сколько таких платежей.
prices = q("""
    select s.tariff_id,
           case when p.paid_at < cast(:cut as timestamptz) then '1. до' else '2. после' end as period,
           p.amount,
           count(*) as n
    from prime.payments p
    join prime.subscriptions s on s.subscription_id = p.subscription_id
    where p.status = 'success'
      and p.paid_at >= cast(:a as timestamptz) and p.paid_at < cast(:b as timestamptz)
    group by 1, 2, 3
""", cut=PRICE_CHANGE, a=BEFORE[0], b=AFTER[1])

prices["tariff"] = prices["tariff_id"].map(TARIFF)
prices["share_%"] = (prices["n"] / prices.groupby(["tariff", "period"])["n"].transform("sum") * 100).round(2)

# Три самые частые суммы в каждой клетке «тариф × период».
(prices.sort_values("n", ascending=False)
       .groupby(["tariff", "period"]).head(3)
       .sort_values(["tariff", "period", "n"], ascending=[True, True, False])
       [["tariff", "period", "amount", "n", "share_%"]])

### Результат аудита

Видим, что отсутствуют данные по двум датам в апреле - это я уже выяснил, когда делал ДЗ. Кроме того, стоит отметить, что количество дней c данными в двух наших отрезках получилось равным - по 90 дней (если вычесть из второго отрезка 2 пропущенных дня). Однако, поскольку продолжительность самих месяцев все равно разная, нормализация с приведением показателей к одному дню оправдана.

Показатель `share_%` указывает, что все платежи до и после cut однородны, следовательно, цена действительно изменилась 1 марта.

Из второй части аудита видно, что количество платежей по обоим тарифам, и по "Базовому", и по "Расширенному" увеличилось после изменения цен. Следовательно, база не сократилась - это частично опровергает H1. Далее проверим, увеличился ли отток.

---
## 3. Месячная таблица: зерно «месяц × тариф»

#### Почему такое зерно:
Наш бизнес-вопрос сформулирован вокруг разницы результатов по двум тарифам: рост цен был разным для двух тарифов и нам нужно понять, повлияло ли это на поведение наших подписчиков. Нас интересует именно финансовый результат нашего продукта в целом, а не результат по усредненному клиенту - поэтому тариф - оптимальная единица.

Мы знаем, что изменеие цен произошло 1 марта и ведем отчет от этой даты. Платеж происходит ежемесячно. Поэтому месяц - оптимальная единица времени для сравнения базы клиентов и оттока.


Одна строка - один месяц одного тарифа. В строке:

| Колонка | Данные |
|---|---|
| `active` | подписки, активные на протяжении месяца |
| `active_start` | подписки, активные на первое число месяца — показатель оттока |
| `churned` | из `active_start` - сколько закончились в этом месяце |
| `started` | новые подписки, начавшиеся в этом месяце |
| `n_payments`, `revenue` | число успешных платежей и выручка |

Таблицу участников `subscription_members` не трогаем: если присоединить платежи к участникам,
один платёж размножится на всех людей подписки и возникнет двойной счет.

In [ ]:
monthly = q("""
    with months as (
        select m.month_start
        from generate_series(cast(:first as timestamptz), cast(:last as timestamptz), interval '1 month') as m(month_start)
    ),
    act as (
        select to_char(m.month_start, 'YYYY-MM') as month,
               s.tariff_id,
               count(*) as active,
               count(*) filter (where s.started_at < m.month_start) as active_start,
               count(*) filter (where s.started_at < m.month_start
                                  and s.ended_at < m.month_start + interval '1 month') as churned,
               count(*) filter (where s.started_at >= m.month_start) as started
        from months m
        join prime.subscriptions s
          on s.started_at < m.month_start + interval '1 month'
         and (s.ended_at is null or s.ended_at >= m.month_start)
        group by 1, 2
    ),
    rev as (
        select to_char(p.paid_at, 'YYYY-MM') as month,
               s.tariff_id,
               count(*) as n_payments,
               sum(p.amount) as revenue
        from prime.payments p
        join prime.subscriptions s on s.subscription_id = p.subscription_id
        where p.status = 'success'
          and p.paid_at >= cast(:first as timestamptz)
          and p.paid_at <  cast(:last as timestamptz) + interval '1 month'
        group by 1, 2
    )
    select a.month, a.tariff_id, a.active, a.active_start, a.churned, a.started,
           coalesce(r.n_payments, 0) as n_payments,
           coalesce(r.revenue, 0) as revenue
    from act a
    left join rev r on r.month = a.month and r.tariff_id = a.tariff_id
    order by 1, 2
""", first=FIRST_MONTH, last=LAST_MONTH)

print(len(monthly), "строк — ожидаем 11 месяцев × 2 тарифа = 22")
monthly.head()

In [ ]:
# Проверим на потерю и двойной счёт денег: выручка в нашей таблице обязана совпасть с суммой успешных платежей, посчитанной напрямую по таблице payments, без соединений.
control = q("""
    select sum(amount) as revenue
    from prime.payments
    where status = 'success'
      and paid_at >= cast(:first as timestamptz)
      and paid_at <  cast(:last as timestamptz) + interval '1 month'
""", first=FIRST_MONTH, last=LAST_MONTH).iloc[0, 0]

diff = monthly["revenue"].sum() - control
print(f"выручка в таблице: {monthly['revenue'].sum():,.2f} ₽".replace(",", " "))
print(f"контроль по payments: {control:,.2f} ₽".replace(",", " "))
assert abs(diff) < 0.01, f"деньги не сходятся на {diff:.2f} ₽"
print("ДЕНЬГИ сходятся: двойного счёта и потерь нет")

# Каждый месяц обязан встретиться ровно по разу на тариф.
assert not monthly.duplicated(["month", "tariff_id"]).any(), "дубли в зерне месяц × тариф"

#### Теперь можем посчитать производные метрики с пересчетом на день ####
`revenue_per_day` - выручка за 1 день (учитываются только дни с данными)

`churn_30d` - отток за 30 дней в процентах - доля тех, кто ушел от общего числа активных пользователей на начало месяца, приведенная к 30 дням

`new_per_day` - количество новых подписок за 1 день

In [ ]:
monthly["tariff"] = monthly["tariff_id"].map(TARIFF)
monthly["days"] = pd.to_datetime(monthly["month"]).dt.days_in_month   
monthly["days_data"] = monthly["month"].map(days_with_data)           

monthly["revenue_per_day"] = monthly["revenue"] / monthly["days_data"]

monthly["churn_30d"] = monthly["churned"] / monthly["active_start"] / monthly["days"] * 30 * 100

monthly["new_per_day"] = monthly["started"] / monthly["days"]

monthly[["month", "tariff", "active", "revenue_per_day", "churn_30d", "new_per_day"]].round(2)